[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_07_research_capstone/ch34_error_analysis_fairness_calibration/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 34 Coding Exam — Subgroup Fairness, Mitigation & Clinical Drift Monitoring (100 Points)

**Book Reference:** Chapter 34, Stage 7 (Coding Exam)  
**Suggested Time:** 50–60 minutes (Open Book, Closed Solution)

| Task | Responsible Clinical AI Audit Component | Points |
| :--- | :--- | :---: |
| **Task 34.1** | `audit_rural_urban_fairness_mch(mch_df, seed=42)` (Leak-free MCH pipeline + Rural/Urban audit) | 30 |
| **Task 34.2** | `mitigate_equal_opportunity_gap(...)` (Validation-calibrated group thresholds -> test EO check) | 35 |
| **Task 34.3** | `clinical_shift_monitoring_report(baseline_df, current_df, feature_cols)` (Multi-feature PSI gate) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 34 Exam Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

tutor.activate(chapter="ch34", notebook="coding_exam")


---
## Exam Task 34.1 (30 Points): `audit_rural_urban_fairness_mch(mch_df, seed=42)`

Audit Rural vs. Urban fairness on `mch_survey_sample` (without mutating `mch_df`):
1. Copy `mch_df`, clean `residence` casing (`str.strip().str.title()`), replace sentinel `anc_visits == 999` with `np.nan`, encode `is_urban = (residence == "Urban").astype(int)`, and map `wealth_quintile` (`Poorest`->1 .. `Richest`->5) into `wealth_rank`.
2. Drop rows missing any value in `["child_stunted", "residence", "maternal_age_years", "anc_visits", "maternal_hb_g_dl", "wealth_rank", "is_urban"]`.
3. Run a stratified 70/30 split (`test_size=0.30, stratify=y, random_state=seed`) keeping `X`, `y`, and `residence` aligned.
4. Fit `Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=500))])` on `(X_train, y_train)`.
5. Select Youden's $J$ threshold **strictly on the training split** `(y_train, prob_train)` via `roc_curve`.
6. Evaluate the held-out test set across `"Rural"` vs. `"Urban"` to produce `subgroup_table` (`["group", "n", "prevalence", "roc_auc", "tpr", "fpr", "brier"]`) and `fairness_gaps` (`demographic_parity_diff`, `equal_opportunity_diff`, `max_fpr_diff`, `equalized_odds_diff`), returning `{"overall_auc": round4, "threshold": round4, "subgroup_table": DataFrame, "fairness_gaps": dict}`.

In [ ]:
def audit_rural_urban_fairness_mch(mch_df, seed=42):
    """Train leak-free stunting model and audit Rural vs Urban fairness."""
    work = mch_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["anc_visits"] = work["anc_visits"].replace(999, np.nan)
    work["is_urban"] = (work["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    work["wealth_rank"] = work["wealth_quintile"].map(w_map)
    fcols = [
        "maternal_age_years", "anc_visits", "maternal_hb_g_dl",
        "wealth_rank", "is_urban",
    ]
    clean = work[["child_stunted", "residence"] + fcols].dropna()
    X = clean[fcols]
    y = clean["child_stunted"].astype(int)
    g = clean["residence"]
    X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
        X, y, g, test_size=0.30, stratify=y, random_state=seed
    )
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500)),
    ])
    pipe.fit(X_tr, y_tr)
    prob_tr = pipe.predict_proba(X_tr)[:, 1]
    prob_te = pipe.predict_proba(X_te)[:, 1]
    fpr_tr, tpr_tr, th_tr = roc_curve(y_tr, prob_tr)
    best_t = float(th_tr[int(np.argmax(tpr_tr - fpr_tr))])
    pred_te = (prob_te >= best_t).astype(int)

    yt = y_te.to_numpy(dtype=int)
    gr = g_te.to_numpy()
    rows, sels, tprs, fprs = [], [], [], []
    for grp in sorted(pd.unique(gr)):
        m = gr == grp
        yt_g, pr_g, yp_g = yt[m], prob_te[m], pred_te[m]
        auc = (
            round(float(roc_auc_score(yt_g, pr_g)), 4)
            if len(np.unique(yt_g)) == 2
            else np.nan
        )
        tp = int(np.sum((yt_g == 1) & (yp_g == 1)))
        fn = int(np.sum((yt_g == 1) & (yp_g == 0)))
        fp = int(np.sum((yt_g == 0) & (yp_g == 1)))
        tn = int(np.sum((yt_g == 0) & (yp_g == 0)))
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        sels.append(float(np.mean(yp_g == 1)))
        tprs.append(tpr)
        fprs.append(fpr)
        rows.append({
            "group": str(grp),
            "n": int(np.sum(m)),
            "prevalence": round(float(np.mean(yt_g == 1)), 4),
            "roc_auc": auc,
            "tpr": round(tpr, 4),
            "fpr": round(fpr, 4),
            "brier": round(float(brier_score_loss(yt_g, pr_g)), 4),
        })
    sub_cols = [
        "group", "n", "prevalence", "roc_auc", "tpr", "fpr", "brier",
    ]
    eo = max(tprs) - min(tprs)
    fpr_d = max(fprs) - min(fprs)
    gaps = {
        "demographic_parity_diff": round(max(sels) - min(sels), 4),
        "equal_opportunity_diff": round(eo, 4),
        "max_fpr_diff": round(fpr_d, 4),
        "equalized_odds_diff": round(max(eo, fpr_d), 4),
    }
    return {
        "overall_auc": round(float(roc_auc_score(yt, prob_te)), 4),
        "threshold": round(best_t, 4),
        "subgroup_table": pd.DataFrame(rows, columns=sub_cols),
        "fairness_gaps": gaps,
    }


---
## Exam Task 34.2 (35 Points): `mitigate_equal_opportunity_gap(y_true_val, y_prob_val, g_val, y_true_test, y_prob_test, g_test, target_tpr=0.75)`

Mitigate Equal Opportunity disparity via validation-only post-processing:
1. Evaluate the uniform `0.5` cutoff on `(y_true_test, y_prob_test, g_test)` to get `before_eo_diff`.
2. Fit group-specific thresholds **strictly on validation** `(y_true_val, y_prob_val, g_val)` selecting the highest unique probability per group achieving `tpr >= target_tpr` (fallback `0.0`), rounded to 4 decimals.
3. Apply the learned `group_thresholds` to the held-out test set to compute `after_eo_diff`.
4. Return `{"before_eo_diff": round4, "after_eo_diff": round4, "group_thresholds": dict, "gap_reduced": bool}` where `gap_reduced = bool(after_eo_diff < before_eo_diff)`.

In [ ]:
def mitigate_equal_opportunity_gap(
    y_true_val, y_prob_val, g_val,
    y_true_test, y_prob_test, g_test,
    target_tpr=0.75,
):
    """Fit validation group thresholds and measure test EO reduction."""
    def _eo_gap(yt_in, yp_in, g_in):
        yt_a = np.asarray(yt_in, dtype=int)
        yp_a = np.asarray(yp_in, dtype=int)
        gr_a = np.asarray(g_in)
        tprs = []
        for g in sorted(pd.unique(gr_a)):
            m = gr_a == g
            tp = int(np.sum((yt_a[m] == 1) & (yp_a[m] == 1)))
            fn = int(np.sum((yt_a[m] == 1) & (yp_a[m] == 0)))
            tprs.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        return round(max(tprs) - min(tprs), 4)

    yt_v = np.asarray(y_true_val, dtype=int)
    yp_v = np.asarray(y_prob_val, dtype=float)
    gr_v = np.asarray(g_val)
    grp_th = {}
    for g in sorted(pd.unique(gr_v)):
        m = gr_v == g
        yt_g, yp_g = yt_v[m], yp_v[m]
        n_pos = int(np.sum(yt_g == 1))
        best_t = 0.0
        if n_pos > 0:
            for t in np.sort(np.unique(yp_g))[::-1]:
                if np.sum((yt_g == 1) & (yp_g >= t)) / n_pos >= target_tpr:
                    best_t = float(t)
                    break
        grp_th[g.item() if hasattr(g, "item") else g] = round(best_t, 4)

    yp_te = np.asarray(y_prob_test, dtype=float)
    gr_te = np.asarray(g_test)
    before_eo = _eo_gap(y_true_test, (yp_te >= 0.5).astype(int), gr_te)
    pred_after = np.array(
        [
            int(p >= grp_th.get(g.item() if hasattr(g, "item") else g, 0.5))
            for p, g in zip(yp_te, gr_te)
        ],
        dtype=int,
    )
    after_eo = _eo_gap(y_true_test, pred_after, gr_te)
    return {
        "before_eo_diff": before_eo,
        "after_eo_diff": after_eo,
        "group_thresholds": grp_th,
        "gap_reduced": bool(after_eo < before_eo),
    }


---
## Exam Task 34.3 (35 Points): `clinical_shift_monitoring_report(baseline_df, current_df, feature_cols)`

Build a production covariate-shift monitoring gate across multiple clinical features:
1. For every column in `feature_cols`, compute PSI (10 quantile bins from baseline with `-inf`/`+inf` outer edges, `eps=1e-6`, clipped & normalized) and classify `"Stable"` (`< 0.10`), `"Moderate Drift"` (`< 0.25`), or `"Severe Drift"` (`>= 0.25`).
2. Assemble `feature_psi` as a `pd.DataFrame` with exact columns `["feature", "psi", "drift_status"]`.
3. Compute `max_psi` (rounded to 4 decimals) and `retrain_recommended = bool(max_psi >= 0.25)`.
4. Return `{"feature_psi": DataFrame, "max_psi": round4, "retrain_recommended": bool}` without mutating input DataFrames.

In [ ]:
def clinical_shift_monitoring_report(baseline_df, current_df, feature_cols):
    """Monitor multi-feature covariate shift with PSI and retrain gate."""
    rows = []
    for col in feature_cols:
        exp = baseline_df[col].dropna().to_numpy(dtype=float)
        act = current_df[col].dropna().to_numpy(dtype=float)
        edges = np.unique(np.quantile(exp, np.linspace(0.0, 1.0, 11)))
        if len(edges) < 2:
            edges = np.array([-np.inf, np.inf])
        else:
            edges = edges.copy()
            edges[0] = -np.inf
            edges[-1] = np.inf
        e_cnt, _ = np.histogram(exp, bins=edges)
        a_cnt, _ = np.histogram(act, bins=edges)
        p_e = np.clip(e_cnt / e_cnt.sum(), 1e-6, None)
        p_e = p_e / p_e.sum()
        p_a = np.clip(a_cnt / a_cnt.sum(), 1e-6, None)
        p_a = p_a / p_a.sum()
        psi = round(float(np.sum((p_a - p_e) * np.log(p_a / p_e))), 4)
        if psi < 0.10:
            status = "Stable"
        elif psi < 0.25:
            status = "Moderate Drift"
        else:
            status = "Severe Drift"
        rows.append({"feature": col, "psi": psi, "drift_status": status})
    psi_df = pd.DataFrame(rows, columns=["feature", "psi", "drift_status"])
    max_psi = round(float(psi_df["psi"].max()), 4) if len(psi_df) else 0.0
    return {
        "feature_psi": psi_df,
        "max_psi": max_psi,
        "retrain_recommended": bool(max_psi >= 0.25),
    }


In [ ]:
exam_submissions = {
    "ch34_exam_01": audit_rural_urban_fairness_mch,
    "ch34_exam_02": mitigate_equal_opportunity_gap,
    "ch34_exam_03": clinical_shift_monitoring_report,
}

tutor.check_exam("ch34", exam_submissions)


---
## Chapter 34 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 34.1 (Leak-Free Threshold Selection):** Selecting Youden's $J$ on `(y_train, prob_train)` keeps the held-out test split strictly untouched while still exposing the stark Rural (`TPR = 0.7333`) vs. Urban (`TPR = 0.0000`) disparity.
- **Task 34.2 (Validation-Only Post-Processing):** Fitting group thresholds on `y_true_test` would overfit the audit itself and fabricate fake fairness gains. Fitting on a separate validation set and evaluating on an untouched test split proves genuine out-of-sample gap reduction.
- **Task 34.3 (Early Drift Detection):** Monitoring `max_psi >= 0.25` across input features triggers a retraining alert immediately when clinic access (`anc_visits`) or maternal nutrition (`maternal_hb_g_dl`) shifts, long before delayed child stunting outcomes can be measured.